In [14]:
import torch
import pandas as pd
import plotly.express as px

import numpy as np
import plotly.io as pio
from IPython.display import display, HTML

# good default for classic notebook / many Jupyter setups
pio.renderers.default = "iframe"

import sys      
sys.path.insert(0, "/storage/project/r-aivanova7-0/shared/eyas/geometry_of_truth_replication")
from src.activations import load_acts
from src.pca import run_pca

In [5]:
# ── Configuration ─────────────────────────────────────────────────────────────
MODEL   = "gemma-2-9b"
LAYER   = 20
DATASET_NAME = "cities"
ACTS_DIR     = "/storage/home/hcoda1/7/eayesh3/scratch/geometry_of_truth/acts"
DATASET_CSV = f"../datasets/{DATASET_NAME}.csv"

In [6]:
# load activations
city_acts = load_acts(model_name = MODEL,
                            dataset_name=DATASET_NAME,
                             layer=LAYER,
                             output_dir=ACTS_DIR)
neg_city_acts = load_acts(model_name = MODEL,
                            dataset_name="neg_cities",
                             layer=LAYER,
                             output_dir=ACTS_DIR)
larger_than_acts = load_acts(model_name = MODEL,
                            dataset_name="larger_than",
                             layer=LAYER,
                             output_dir=ACTS_DIR)
                             

In [50]:
joint_city_acts = np.concat((city_acts,neg_city_acts, larger_than_acts),axis=0)

In [51]:
joint_city_pca = run_pca(torch.from_numpy(joint_city_acts),k=50)

In [52]:
proj = joint_city_pca.projections.numpy()   # [n_statements, n_pcs]
ev   = joint_city_pca.explained_var_ratio

print(f"Projections shape: {proj.shape}")
print(f"PC1: {ev[0]:.2%}  PC2: {ev[1]:.2%}  PC3: {ev[2]:.2%}")

cumsum = np.cumsum(ev)
print(f"{np.argmax(cumsum > 0.95)+1} components until 95% var explained")

Projections shape: (4972, 50)
PC1: 19.49%  PC2: 14.52%  PC3: 8.57%
1 components until 95% var explained


In [53]:
# ── Load dataset ──────────────────────────────────────────────────────────────
df_cities = pd.read_csv(f"../datasets/cities.csv")
df_larger_than = pd.read_csv(f"../datasets/larger_than.csv")
df = pd.concat([df_cities, df_cities, df_larger_than],ignore_index=True, join="outer") 
assert len(df) == proj.shape[0], f"Row count mismatch: {len(df)} vs {proj.shape[0]}"

df["PC1"]   = proj[:, 0]
df["PC2"]   = proj[:, 1]
df["PC3"]   = proj[:, 2]
df["truth"] = df["label"].map({1: "True", 0: "False"})

In [8]:
# ── 2D scatter ────────────────────────────────────────────────────────────────
fig = px.scatter(
    df,
    x="PC1", y="PC2",
    color="truth",
    color_discrete_map={"True": "#d62728", "False": "#1f77b4"},
    hover_data={"statement": True, "PC1": False, "PC2": False},
    title=f"{MODEL} — cities & Larget than — layer {LAYER}  |  PC1: {ev[0]:.1%}, PC2: {ev[1]:.1%}",
    template="plotly_white",
    opacity=0.7,
)
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.update_traces(marker_size=5)
fig.show()

NameError: name 'df' is not defined

In [11]:
import torch                                                                        
import pandas as pd
from glob import glob                                                               
import os       

acts_dir = "/storage/home/hcoda1/7/eayesh3/scratch/geometry_of_truth/acts_original" 
model_name = "llama-2-13b-hf"
dataset_name = "cities"                                                             
layer = 16      
BATCH_SIZE = 64  # whatever was used                                                

# Load the raw saved chunks (before concatenation)                                  
directory = os.path.join(acts_dir, model_name, dataset_name)
files = sorted(                                                                     
  glob(os.path.join(directory, f"layer_{layer}_*.pt")),
  key=lambda f: int(os.path.basename(f).split("_")[-1].replace(".pt", ""))        
)               

# Load statements to check token lengths
df = pd.read_csv(f"../datasets/{dataset_name}.csv")                                    
statements = df["statement"].tolist()                                               

# For each batch, compare norms of shortest vs longest statement                    
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-13b-hf")              

for file_idx, f in enumerate(files[:30]):  # check first few batches                 
  batch_start = file_idx * BATCH_SIZE                                             
  batch = statements[batch_start : batch_start + BATCH_SIZE]                      
  lengths = [len(tokenizer.encode(s)) for s in batch]
  acts = torch.load(f, weights_only=True).float()                                 

  norms = acts.norm(dim=1)                                                        
  shortest_idx = lengths.index(min(lengths))                                      
  longest_idx  = lengths.index(max(lengths))                                      

  print(f"Batch {file_idx}: lengths {min(lengths)}–{max(lengths)}")               
  print(f"  Norm of shortest (idx {shortest_idx}): {norms[shortest_idx]:.2f}")
  print(f"  Norm of longest  (idx {longest_idx}):  {norms[longest_idx]:.2f}")     
  print(f"  Median norm: {norms.median():.2f}")                                   
  print()

Batch 0: lengths 9–18
  Norm of shortest (idx 34): 39.34
  Norm of longest  (idx 12):  39.28
  Median norm: 39.97

Batch 1: lengths 9–16
  Norm of shortest (idx 38): 40.57
  Norm of longest  (idx 61):  38.61
  Median norm: 39.73

Batch 2: lengths 9–18
  Norm of shortest (idx 62): 39.63
  Norm of longest  (idx 9):  39.41
  Median norm: 39.68

Batch 3: lengths 10–19
  Norm of shortest (idx 0): 40.59
  Norm of longest  (idx 49):  39.38
  Median norm: 39.97

Batch 4: lengths 10–20
  Norm of shortest (idx 1): 37.11
  Norm of longest  (idx 16):  40.03
  Median norm: 40.03

Batch 5: lengths 10–16
  Norm of shortest (idx 34): 40.06
  Norm of longest  (idx 22):  38.80
  Median norm: 39.70

Batch 6: lengths 9–17
  Norm of shortest (idx 48): 39.79
  Norm of longest  (idx 54):  42.06
  Median norm: 39.66

Batch 7: lengths 10–19
  Norm of shortest (idx 3): 40.64
  Norm of longest  (idx 50):  38.85
  Median norm: 39.64

Batch 8: lengths 10–19
  Norm of shortest (idx 0): 41.41
  Norm of longest  (idx

In [12]:
import sys                                                                          
sys.path.insert(0, "../geometry-of-truth")                                             
   

In [13]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-13b-hf")
print(tokenizer.padding_side)  # expected: "right"                                  



right


In [ ]:
             
from src.models import load_model
model, _ = load_model("llama-2-13b")                                                
print(model.tokenizer.padding_side)  # should be "left" — set explicitly in         


Loading LLaMA-2-13B from HuggingFace...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Wrapping with TransformerLens...
